# Stable Diffusion on Pokemon

## From Pretrained Text-to-Image to Pokemon Art Style

In this notebook, we **fine-tune Stable Diffusion 1.5** on the Pokemon BLIP Captions dataset using **LoRA (Low-Rank Adaptation)**. Instead of training a model from scratch, we leverage a model pretrained on 2 billion image-text pairs and efficiently adapt it to generate Pokemon-style images.

### Stable Diffusion Architecture

```
                        ┌─────────────────────────────────────────────────┐
                        │           Stable Diffusion Pipeline             │
                        │                                                 │
  "a blue pokemon"      │  ┌──────────┐                                  │
  ──────────────────────┼─►│  CLIP     │  text features                  │
                        │  │  Text     │──────────┐                      │
                        │  │  Encoder  │          │ cross-attention       │
                        │  └──────────┘          ▼                       │
                        │                  ┌───────────┐                 │
  Random noise          │                  │   U-Net   │  denoised       │   ┌───────┐
  (64×64×4 latent) ─────┼─────────────────►│  + LoRA   │──latent────────┼──►│  VAE   │──► 512×512
                        │                  │  adapters │                 │   │Decoder │    image
                        │    timestep ────►│           │                 │   └───────┘
                        │                  └───────────┘                 │
                        └─────────────────────────────────────────────────┘
```

| Component | Role |
|-----------|------|
| **VAE** (Variational Autoencoder) | Compresses 512×512 images into 64×64×4 latent space (8× compression). Diffusion happens in this compressed space — much cheaper than pixel space! |
| **CLIP Text Encoder** | Converts text prompts into feature vectors that guide the U-Net via cross-attention. Frozen (not trained). |
| **U-Net** | The denoiser — predicts and removes noise from latent representations, conditioned on text. This is what we fine-tune with LoRA. |
| **Noise Scheduler** | Controls the noise schedule (same concept as DDPM). |

### Why Fine-Tune Instead of Train from Scratch?

Training a diffusion model from scratch on ~800 images produces poor results. Stable Diffusion was pretrained on **2 billion** image-text pairs — it already knows how to generate realistic images. We just need to teach it the **Pokemon style** using **LoRA**, which:
- Trains only **~1-4M parameters** instead of all 860M
- Takes **~15-30 minutes** on a single GPU
- Produces **high-quality 512×512 Pokemon images**

In [ ]:
# ============================================================
# Setup: Install and import dependencies
# ============================================================
# !pip install diffusers transformers accelerate datasets peft torch torchvision matplotlib numpy

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image as PILImage

# Device setup
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 1: Load the Pokemon Dataset

We use the **Pokemon BLIP Captions** dataset — it's ideal for teaching because:
- Captions are simple and visually grounded ("a blue pokemon with large eyes")
- Stylized images mean even a small model can produce recognizable outputs
- Students can quickly see whether conditioning is working

In [ ]:
# ============================================================
# Step 1: Load and Explore the Pokemon Dataset
# ============================================================
from datasets import load_dataset

# Load Pokemon BLIP Captions dataset
print("Loading Pokemon BLIP Captions dataset...")
pokemon_dataset = load_dataset("reach-vb/pokemon-blip-captions", split="train")
print(f"Dataset size: {len(pokemon_dataset)} images")
print(f"Features: {pokemon_dataset.features}")
print(f"\nExample captions:")
for i in range(5):
    print(f"  [{i}] {pokemon_dataset[i]['text']}")

In [ ]:
# ============================================================
# Visualization: Dataset Gallery
# ============================================================

fig, axes = plt.subplots(3, 6, figsize=(18, 10))

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(len(pokemon_dataset))
    item = pokemon_dataset[idx]
    img = item['image']
    caption = item['text']

    ax.imshow(img)
    ax.set_title(caption[:40] + ('...' if len(caption) > 40 else ''),
                fontsize=8, wrap=True)
    ax.axis('off')

plt.suptitle('Pokemon BLIP Captions Dataset — Sample Gallery',
            fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Step 1b: Create PyTorch Dataset with Preprocessing
# ============================================================
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

IMAGE_SIZE = 512  # Native resolution for Stable Diffusion 1.5

# Image preprocessing: resize to 512x512, normalize to [-1, 1]
image_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])


class PokemonCaptionDataset(Dataset):
    """Wraps the HuggingFace dataset into a PyTorch Dataset."""
    def __init__(self, hf_dataset, transform):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = self.transform(item['image'].convert('RGB'))
        caption = item['text']
        return image, caption


# Create a 90/10 train/test split
n_total = len(pokemon_dataset)
n_test = max(10, n_total // 10)
n_train = n_total - n_test

pokemon_train = pokemon_dataset.select(range(n_train))
pokemon_test = pokemon_dataset.select(range(n_train, n_total))

train_dataset = PokemonCaptionDataset(pokemon_train, image_transform)
test_dataset = PokemonCaptionDataset(pokemon_test, image_transform)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True,
                         num_workers=0, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False,
                        num_workers=0, drop_last=True)

# Verify a batch
images, captions = next(iter(train_loader))
print(f"Image batch shape: {images.shape}  (batch, channels, height, width)")
print(f"Image value range: [{images.min():.1f}, {images.max():.1f}]  (normalized to [-1, 1])")
print(f"Caption example: '{captions[0]}'")
print(f"\nTrain: {len(train_dataset)} images, Test: {len(test_dataset)} images at {IMAGE_SIZE}x{IMAGE_SIZE}")

## The VAE: Latent Space Compression

Stable Diffusion does **not** work directly on pixels. Instead, a pretrained VAE compresses images into a much smaller **latent space**:

$$x \in \mathbb{R}^{512 \times 512 \times 3} \longrightarrow z = \mathcal{E}(x) \in \mathbb{R}^{64 \times 64 \times 4}$$

This 8× spatial compression makes diffusion ~64× cheaper. After sampling, the decoder maps the clean latent back to pixels:

$$\hat{x} = \mathcal{D}(z_0)$$

In [ ]:
# ============================================================
# Visualization: VAE Encode / Decode Demo
# ============================================================
# Stable Diffusion works in LATENT space, not pixel space.
# The VAE compresses 512x512 RGB images into 64x64x4 latent representations.

from diffusers import AutoencoderKL

# Load just the VAE for demonstration
vae_demo = AutoencoderKL.from_pretrained(
    "runwayml/stable-diffusion-v1-5", subfolder="vae"
).to(device)
vae_demo.eval()
vae_demo.requires_grad_(False)

# Take a few images from the dataset
n_demo = 4
demo_images = []
demo_captions = []
for idx in [0, 50, 100, 200]:
    item = pokemon_dataset[idx]
    img = image_transform(item['image'].convert('RGB'))
    demo_images.append(img)
    demo_captions.append(item['text'])
demo_batch = torch.stack(demo_images).to(device)

# Encode to latent space, then decode back
with torch.no_grad():
    latents = vae_demo.encode(demo_batch).latent_dist.sample()
    latents_scaled = latents * vae_demo.config.scaling_factor
    decoded = vae_demo.decode(latents_scaled / vae_demo.config.scaling_factor).sample

print(f"Original images:  {demo_batch.shape}  (batch, 3, 512, 512)")
print(f"Latent space:     {latents.shape}  (batch, 4, 64, 64)  -- 8x spatial compression!")
print(f"Decoded images:   {decoded.shape}  (batch, 3, 512, 512)")
print(f"\nCompression ratio: {512*512*3 / (64*64*4):.1f}x fewer values in latent space")

fig, axes = plt.subplots(3, n_demo, figsize=(5 * n_demo, 14))

for i in range(n_demo):
    # Original
    orig = (demo_batch[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
    axes[0, i].imshow(orig)
    axes[0, i].set_title(demo_captions[i][:35], fontsize=9)
    axes[0, i].axis('off')

    # Latent (show first 3 of 4 channels as RGB-ish visualization)
    lat_viz = latents[i, :3].cpu().permute(1, 2, 0).numpy()
    lat_viz = (lat_viz - lat_viz.min()) / (lat_viz.max() - lat_viz.min() + 1e-8)
    axes[1, i].imshow(lat_viz)
    axes[1, i].set_title(f'Latent: {latents.shape[2]}x{latents.shape[3]}x{latents.shape[1]}', fontsize=10)
    axes[1, i].axis('off')

    # Reconstructed
    recon = (decoded[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
    axes[2, i].imshow(recon)
    axes[2, i].set_title('VAE Reconstruction', fontsize=10)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original\n512x512', fontsize=12, rotation=0, labelpad=60, va='center')
axes[1, 0].set_ylabel('Latent\n64x64x4', fontsize=12, rotation=0, labelpad=60, va='center')
axes[2, 0].set_ylabel('Decoded\n512x512', fontsize=12, rotation=0, labelpad=60, va='center')

plt.suptitle('VAE: Image → Latent → Reconstructed Image\nDiffusion happens in the compressed 64x64x4 latent space',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

del vae_demo
if device.type == 'cuda':
    torch.cuda.empty_cache()

## Step 2: Load Pretrained Stable Diffusion 1.5

We load the full Stable Diffusion 1.5 pipeline using the HuggingFace `diffusers` library. This one call downloads all four components:

1. **`vae`** — compresses images to/from latent space (frozen, not trained)
2. **`text_encoder`** + **`tokenizer`** — converts text to CLIP embeddings (frozen, not trained)
3. **`unet`** — the denoiser that we will fine-tune with LoRA
4. **`scheduler`** — controls the diffusion noise schedule

All components except the U-Net's LoRA adapters remain frozen during fine-tuning.

In [ ]:
# ============================================================
# Step 2: Load Pretrained Stable Diffusion 1.5
# ============================================================
from diffusers import StableDiffusionPipeline, DDPMScheduler

MODEL_ID = "runwayml/stable-diffusion-v1-5"

print(f"Loading Stable Diffusion 1.5 from: {MODEL_ID}")
print("This may take a minute on first download (~4GB)...\n")

# One call downloads the VAE, text encoder, U-Net, tokenizer, and scheduler
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32
)

# Extract the four core components
vae = pipe.vae.to(device)
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder.to(device)
unet = pipe.unet.to(device)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

# Freeze VAE and text encoder — we only fine-tune the U-Net (via LoRA)
vae.requires_grad_(False)
vae.eval()
text_encoder.requires_grad_(False)
text_encoder.eval()

# Print component summary
print("Stable Diffusion 1.5 Components:")
print("=" * 55)
vae_params = sum(p.numel() for p in vae.parameters())
te_params = sum(p.numel() for p in text_encoder.parameters())
unet_params = sum(p.numel() for p in unet.parameters())
print(f"  VAE:            {vae_params:>12,} params (frozen)")
print(f"  Text Encoder:   {te_params:>12,} params (frozen)")
print(f"  U-Net:          {unet_params:>12,} params (will add LoRA)")
print(f"  {'─'*43}")
print(f"  Total:          {vae_params+te_params+unet_params:>12,} params")
print(f"\n  Text hidden dim: {text_encoder.config.hidden_size}")
print(f"  Latent channels: {vae.config.latent_channels}")
print(f"  Latent scale:    {vae.config.scaling_factor}")

del pipe
if device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Baseline: Generate with vanilla SD 1.5 (BEFORE fine-tuning)
# ============================================================
# We use the high-level StableDiffusionPipeline for inference.
# This handles tokenization, text encoding, denoising loop,
# classifier-free guidance, and VAE decoding automatically.

baseline_pipe = StableDiffusionPipeline(
    vae=vae, text_encoder=text_encoder, tokenizer=tokenizer,
    unet=unet, scheduler=noise_scheduler,
    safety_checker=None, feature_extractor=None,
    requires_safety_checker=False,
)

baseline_prompts = [
    "a drawing of a green pokemon with red eyes",
    "a blue water creature with fins",
    "a red fire-breathing dragon pokemon",
    "a cute yellow electric mouse pokemon",
]

print("Generating baseline images with vanilla SD 1.5 (before fine-tuning)...")
generator = torch.Generator(device=device).manual_seed(42)
baseline_images = baseline_pipe(
    baseline_prompts, num_inference_steps=30, guidance_scale=7.5,
    generator=generator,
).images

fig, axes = plt.subplots(1, len(baseline_prompts), figsize=(5 * len(baseline_prompts), 6))
for i, (img, prompt) in enumerate(zip(baseline_images, baseline_prompts)):
    axes[i].imshow(img)
    axes[i].set_title(prompt[:45], fontsize=9, wrap=True)
    axes[i].axis('off')

plt.suptitle('Baseline: Vanilla SD 1.5 (before Pokemon fine-tuning)\nThe model generates images but not in the Pokemon art style',
            fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

del baseline_pipe

## Step 3: LoRA — Efficient Fine-Tuning

### The Problem with Full Fine-Tuning

The U-Net has **~860M parameters**. Fine-tuning all of them would require enormous GPU memory and risk overfitting on our small 833-image dataset.

### The LoRA Solution

**LoRA (Low-Rank Adaptation)** freezes the original weights and injects small trainable matrices into the attention layers:

$$W' = W + BA$$

where $W$ is the frozen original weight matrix $(d \times d)$, and $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times d}$ are small trainable matrices with rank $r \ll d$.

```
Original attention layer:          With LoRA:

  x ──► [  W  ] ──► y              x ──► [  W (frozen) ] ──► y
         (d×d)                      │                         ▲
                                    └──► [ B ]──►[ A ] ──────┘
                                          (d×r)   (r×d)
                                          trainable!
```

**Key insight:** Most of the "knowledge" needed for Pokemon style can be captured in a low-rank update. With rank r=32, we only train ~6M parameters instead of 860M — less than 1% of the model.

### Cross-Attention: How the U-Net "Reads" Text

```
U-Net spatial features          CLIP text features
(what it's generating)          (what to generate)
        │                              │
        ▼                              ▼
    ┌───────┐                     ┌───────┐
    │  Q    │    attention        │ K, V  │
    │(query)│◄───────────────────►│(key,  │
    │       │   "Which text       │ value)│
    └───┬───┘    tokens matter    └───────┘
        │         for this pixel?"
        ▼
  Updated spatial features
  (now informed by text)
```

At each spatial position, the U-Net asks: *"Which parts of the text description are relevant here?"*

In [ ]:
# ============================================================
# Step 3: Inject LoRA Adapters into the U-Net
# ============================================================
from peft import LoraConfig, get_peft_model

# LoRA configuration: which layers to adapt and with what rank
lora_config = LoraConfig(
    r=32,                             # Rank of the low-rank matrices
    lora_alpha=32,                    # Scaling factor (alpha/r = effective scale)
    init_lora_weights="gaussian",
    target_modules=[                  # Inject LoRA into attention AND feed-forward layers
        "to_k", "to_q", "to_v", "to_out.0",
        "ff.net.0.proj", "ff.net.2",  # Feed-forward layers in transformer blocks
    ],
)

# Inject LoRA adapters into the U-Net
unet = get_peft_model(unet, lora_config)

# Count parameters
total_params = sum(p.numel() for p in unet.parameters())
trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print("LoRA injection complete!")
print("=" * 55)
print(f"  Total U-Net params:     {total_params:>12,}")
print(f"  Frozen params:          {frozen_params:>12,}  ({frozen_params/total_params:.1%})")
print(f"  Trainable LoRA params:  {trainable_params:>12,}  ({trainable_params/total_params:.1%})")
print(f"\n  → We only train {trainable_params/total_params:.2%} of the model!")
print(f"  → LoRA rank = {lora_config.r}, targeting {len(lora_config.target_modules)} layer types")

In [ ]:
# ============================================================
# Visualization: LoRA Concept Diagram
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel 1: Standard vs LoRA weight update ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Standard Fine-Tuning vs LoRA', fontweight='bold', fontsize=13)

# Standard fine-tuning
rect_std = mpatches.FancyBboxPatch((0.3, 6), 4, 3, boxstyle="round,pad=0.15",
                                    facecolor='#e63946', alpha=0.2, edgecolor='#e63946', lw=2)
ax.add_patch(rect_std)
ax.text(2.3, 8.3, 'Standard Fine-Tuning', ha='center', fontsize=11, fontweight='bold')
ax.text(2.3, 7.3, 'Update ALL 860M params', ha='center', fontsize=10)
ax.text(2.3, 6.5, 'Slow, high memory', ha='center', fontsize=9, color='gray')

# LoRA
rect_lora = mpatches.FancyBboxPatch((5.5, 6), 4.2, 3, boxstyle="round,pad=0.15",
                                     facecolor='#2a9d8f', alpha=0.2, edgecolor='#2a9d8f', lw=2)
ax.add_patch(rect_lora)
ax.text(7.6, 8.3, 'LoRA Fine-Tuning', ha='center', fontsize=11, fontweight='bold')
ax.text(7.6, 7.3, f'Update only ~{trainable_params/1e6:.1f}M params', ha='center', fontsize=10)
ax.text(7.6, 6.5, 'Fast, low memory', ha='center', fontsize=9, color='gray')

# Formula
ax.text(5, 4.5, "W' = W + B A", ha='center', fontsize=18, fontweight='bold',
        family='monospace')
ax.text(5, 3.3, 'W: frozen original weights (d × d)', ha='center', fontsize=10)
ax.text(5, 2.5, 'B: trainable (d × r)', ha='center', fontsize=10, color='#2a9d8f')
ax.text(5, 1.8, 'A: trainable (r × d)', ha='center', fontsize=10, color='#2a9d8f')
ax.text(5, 0.8, f'r = {lora_config.r} (rank) << d = 320...1280', ha='center',
        fontsize=10, style='italic', color='gray')

# --- Panel 2: Where LoRA is injected ---
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('LoRA Injection Points in U-Net', fontweight='bold', fontsize=13)

layers = ['to_q (Query)', 'to_k (Key)', 'to_v (Value)', 'to_out (Output)']
colors_l = ['#457b9d', '#e76f51', '#2a9d8f', '#f4a261']
for i, (name, c) in enumerate(zip(layers, colors_l)):
    y = 8 - i * 2
    # Frozen weight box
    rect = mpatches.FancyBboxPatch((0.5, y - 0.5), 3, 1, boxstyle="round,pad=0.1",
                                    facecolor='#ddd', alpha=0.5, edgecolor='gray', lw=1.5)
    ax2.add_patch(rect)
    ax2.text(2, y, 'W (frozen)', ha='center', va='center', fontsize=9, color='gray')

    # Plus sign
    ax2.text(4.2, y, '+', ha='center', va='center', fontsize=16, fontweight='bold')

    # LoRA adapter box
    rect_a = mpatches.FancyBboxPatch((5, y - 0.5), 3.5, 1, boxstyle="round,pad=0.1",
                                      facecolor=c, alpha=0.25, edgecolor=c, lw=2)
    ax2.add_patch(rect_a)
    ax2.text(6.75, y, f'LoRA: {name}', ha='center', va='center', fontsize=9,
            fontweight='bold', color=c)

ax2.text(5, 0.3, f'Applied to every cross-attention & self-attention block in the U-Net',
        ha='center', fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.show()

## Step 4: Classifier-Free Guidance (CFG)

### What is Classifier-Free Guidance?

During **training**, we randomly drop the text conditioning ~10% of the time (replace with empty text). This teaches the model to generate *both* with and without text guidance.

During **inference**, we combine the two predictions:

$$\hat{\epsilon} = \epsilon_\theta(x_t, \varnothing) + s \cdot \left(\epsilon_\theta(x_t, \text{text}) - \epsilon_\theta(x_t, \varnothing)\right)$$

where $s$ is the **guidance scale**:
- $s = 1$: No guidance (use text prediction as-is)
- $s = 3\text{-}6$: Moderate guidance (better text following)
- $s > 7$: Strong guidance (very prompt-faithful, may lose diversity)

In [ ]:
# ============================================================
# Visualization: Classifier-Free Guidance Explained
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Training with CFG dropout
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

ax.text(5, 9, 'Training with CFG Dropout', ha='center', fontsize=13, fontweight='bold')

# 90% path
rect1 = mpatches.FancyBboxPatch((0.5, 5.5), 4, 2.5, boxstyle="round,pad=0.1",
                                  facecolor='#2a9d8f', alpha=0.2, edgecolor='#2a9d8f')
ax.add_patch(rect1)
ax.text(2.5, 7.5, '90% of time:', ha='center', fontsize=10, fontweight='bold')
ax.text(2.5, 6.5, 'Use real caption\n"a blue pokemon..."', ha='center', fontsize=9)

# 10% path
rect2 = mpatches.FancyBboxPatch((5.5, 5.5), 4, 2.5, boxstyle="round,pad=0.1",
                                  facecolor='#e63946', alpha=0.2, edgecolor='#e63946')
ax.add_patch(rect2)
ax.text(7.5, 7.5, '10% of time:', ha='center', fontsize=10, fontweight='bold')
ax.text(7.5, 6.5, 'Use empty text ""\n(unconditional)', ha='center', fontsize=9)

ax.text(5, 4.5, 'Both feed into same U-Net', ha='center', fontsize=10, style='italic')
ax.annotate('', xy=(5, 3.5), xytext=(5, 4.2),
           arrowprops=dict(arrowstyle='->', lw=2))
rect3 = mpatches.FancyBboxPatch((2, 2), 6, 1.5, boxstyle="round,pad=0.1",
                                  facecolor='#457b9d', alpha=0.2, edgecolor='#457b9d')
ax.add_patch(rect3)
ax.text(5, 2.75, 'U-Net learns both modes', ha='center', fontsize=10, fontweight='bold')

# Panel 2: Inference formula
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

ax.text(5, 9, 'Inference with CFG', ha='center', fontsize=13, fontweight='bold')
ax.text(5, 7.5, 'Two forward passes:', ha='center', fontsize=11)
ax.text(5, 6.5, r'$\epsilon_{uncond} = \epsilon_\theta(x_t, \varnothing)$', ha='center', fontsize=12)
ax.text(5, 5.5, r'$\epsilon_{cond} = \epsilon_\theta(x_t, \mathrm{text})$', ha='center', fontsize=12)
ax.text(5, 4, 'Combine:', ha='center', fontsize=11, fontweight='bold')
ax.text(5, 2.8, r'$\hat{\epsilon} = \epsilon_{uncond} + s \cdot (\epsilon_{cond} - \epsilon_{uncond})$',
       ha='center', fontsize=13, color='#e63946',
       bbox=dict(boxstyle='round', facecolor='#fee', edgecolor='#e63946'))
ax.text(5, 1.5, 'guidance scale s controls\nhow much to follow the text',
       ha='center', fontsize=10, style='italic')

# Panel 3: Effect of guidance scale
ax = axes[2]
scales = [1, 3, 5, 7, 10]
quality = [0.3, 0.7, 0.9, 0.85, 0.6]
diversity = [0.9, 0.7, 0.5, 0.3, 0.15]
text_follow = [0.2, 0.5, 0.8, 0.95, 0.99]

ax.plot(scales, quality, 'o-', color='#2a9d8f', linewidth=2, markersize=8, label='Image quality')
ax.plot(scales, diversity, 's-', color='#457b9d', linewidth=2, markersize=8, label='Diversity')
ax.plot(scales, text_follow, '^-', color='#e63946', linewidth=2, markersize=8, label='Text following')
ax.axvline(x=5, color='gray', linestyle='--', alpha=0.5)
ax.text(5.2, 0.95, 'sweet spot', fontsize=9, color='gray')
ax.set_xlabel('Guidance Scale (s)', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Effect of Guidance Scale', fontweight='bold', fontsize=13)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.1)

plt.suptitle('Classifier-Free Guidance: The Key Trick for Text-Conditioned Generation',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Step 5: LoRA Fine-Tuning Training Loop

The training loop follows the standard latent diffusion recipe:

1. **Encode** images to VAE latents (64×64×4) using frozen VAE
2. **Encode** text captions with frozen CLIP (with 10% CFG dropout)
3. **Sample** a random timestep and add noise to the latents
4. **Predict** the noise with the U-Net (conditioned on text via cross-attention)
5. **Compute** MSE loss between predicted and actual noise
6. **Update** only the LoRA parameters

The loss is:

$$\mathcal{L} = \mathbb{E}_{z_0, \epsilon, t, c} \left[ \| \epsilon - \epsilon_\theta(z_t, t, c) \|_2^2 \right]$$

In [ ]:
# ============================================================
# Helper functions for training
# ============================================================

def encode_text(captions, tokenizer, text_encoder, device):
    """Convert text captions into CLIP hidden states for cross-attention."""
    tokens = tokenizer(
        captions,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        text_features = text_encoder(tokens.input_ids)[0]
    return text_features


def encode_images_to_latents(images, vae):
    """Encode pixel-space images into VAE latent space."""
    with torch.no_grad():
        latent_dist = vae.encode(images.to(vae.dtype)).latent_dist
        latents = latent_dist.sample() * vae.config.scaling_factor
    return latents


# Quick test
test_captions = ["a blue pokemon with large eyes"]
test_features = encode_text(test_captions, tokenizer, text_encoder, device)
print(f"Text features shape: {test_features.shape}  (batch, seq_len, hidden_dim={test_features.shape[-1]})")

test_latents = encode_images_to_latents(images[:1].to(device), vae)
print(f"Image latents shape: {test_latents.shape}  (batch, 4, 64, 64)")
print(f"\nThe U-Net will denoise these 64x64x4 latents, conditioned on the {test_features.shape[-1]}-dim text features.")

In [ ]:
# ============================================================
# Step 5: LoRA Fine-Tuning Training Loop
# ============================================================
import math

weight_dtype = torch.float16 if device.type == 'cuda' else torch.float32

# Move frozen models to lower precision for memory efficiency
vae.to(dtype=weight_dtype)
text_encoder.to(dtype=weight_dtype)

# Keep UNet in fp32 for gradient precision on LoRA weights
unet.float()

# Training config
num_steps = 1500
gradient_accumulation_steps = 4
base_lr = 5e-5
warmup_steps = 100
cfg_dropout_prob = 0.10
log_every = 50
sample_every = 300

optimizer = torch.optim.AdamW(
    [p for p in unet.parameters() if p.requires_grad],
    lr=base_lr, weight_decay=1e-4, betas=(0.9, 0.999), eps=1e-8
)

# Cosine LR schedule with linear warmup
def get_lr(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(1, num_steps - warmup_steps)
    return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)

test_prompts = [
    "a drawing of a green pokemon with red eyes",
    "a red fire-breathing dragon pokemon",
    "a cute pink round creature with big eyes",
    "a blue water pokemon with fins",
]

train_losses = []
test_losses = []
lr_history = []
sample_history = {}
global_step = 0

# Pre-compute empty-text embedding for CFG dropout
empty_emb = encode_text([""], tokenizer, text_encoder, device).to(dtype=weight_dtype)

trainable_count = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print("Starting LoRA fine-tuning...")
print(f"  Total steps:          {num_steps}")
print(f"  Grad accumulation:    {gradient_accumulation_steps}")
print(f"  Effective batch size: {train_loader.batch_size * gradient_accumulation_steps}")
print(f"  Base LR:              {base_lr}")
print(f"  LR warmup steps:      {warmup_steps}")
print(f"  CFG dropout:          {cfg_dropout_prob:.0%}")
print(f"  Trainable params:     {trainable_count:,}")
print("=" * 60)

# Build a StableDiffusionPipeline for generating samples during training
# This uses the high-level API instead of manual denoising loops
sample_pipe = StableDiffusionPipeline(
    vae=vae, text_encoder=text_encoder, tokenizer=tokenizer,
    unet=unet, scheduler=noise_scheduler,
    safety_checker=None, feature_extractor=None,
    requires_safety_checker=False,
)

unet.train()
data_iter = iter(train_loader)
optimizer.zero_grad()
running_loss = 0.0

for step in range(num_steps * gradient_accumulation_steps):
    try:
        batch_images, batch_captions = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch_images, batch_captions = next(data_iter)

    batch_images = batch_images.to(device, dtype=weight_dtype)

    # 1. Encode images to latent space
    with torch.no_grad():
        latents = encode_images_to_latents(batch_images, vae)

    # 2. Encode text (with CFG dropout)
    with torch.no_grad():
        if np.random.random() < cfg_dropout_prob:
            text_emb = empty_emb.expand(latents.shape[0], -1, -1)
        else:
            text_emb = encode_text(
                list(batch_captions), tokenizer, text_encoder, device
            ).to(dtype=weight_dtype)

    # Cast to fp32 for UNet forward pass
    latents_f32 = latents.float()
    text_emb_f32 = text_emb.float()

    # 3. Sample noise and timesteps
    noise = torch.randn_like(latents_f32)
    timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                              (latents_f32.shape[0],), device=device).long()

    # 4. Add noise to latents (forward process)
    noisy_latents = noise_scheduler.add_noise(latents_f32, noise, timesteps)

    # 5. Predict the noise
    noise_pred = unet(noisy_latents, timesteps,
                      encoder_hidden_states=text_emb_f32).sample

    # 6. Compute loss and accumulate gradients
    loss = F.mse_loss(noise_pred.float(), noise.float()) / gradient_accumulation_steps
    loss.backward()
    running_loss += loss.item()

    # Update weights after accumulation
    if (step + 1) % gradient_accumulation_steps == 0:
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        global_step += 1

        train_losses.append(running_loss)
        lr_history.append(optimizer.param_groups[0]['lr'])
        running_loss = 0.0

        if global_step % log_every == 0:
            avg_loss = np.mean(train_losses[-log_every:])
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  Step {global_step:>5d}/{num_steps}  |  Train Loss: {avg_loss:.4f}  |  LR: {current_lr:.2e}")

        if global_step % sample_every == 0 or global_step == num_steps:
            # Compute test loss
            unet.eval()
            test_loss_total = 0.0
            test_batches = 0
            max_test_batches = min(10, len(test_loader))
            test_iter = iter(test_loader)
            with torch.no_grad():
                for _ in range(max_test_batches):
                    try:
                        t_imgs, t_caps = next(test_iter)
                    except StopIteration:
                        break
                    t_imgs = t_imgs.to(device, dtype=weight_dtype)
                    t_latents = encode_images_to_latents(t_imgs, vae).float()
                    t_text_emb = encode_text(list(t_caps), tokenizer, text_encoder, device).float()
                    t_noise = torch.randn_like(t_latents)
                    t_timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                               (t_latents.shape[0],), device=device).long()
                    t_noisy = noise_scheduler.add_noise(t_latents, t_noise, t_timesteps)
                    t_pred = unet(t_noisy, t_timesteps, encoder_hidden_states=t_text_emb).sample
                    test_loss_total += F.mse_loss(t_pred.float(), t_noise.float()).item()
                    test_batches += 1
            avg_test_loss = test_loss_total / max(test_batches, 1)
            test_losses.append((global_step, avg_test_loss))
            print(f"         → Test Loss: {avg_test_loss:.4f}")

            # Generate sample images using the high-level pipeline
            # Temporarily cast UNet to weight_dtype for inference compatibility
            unet.to(dtype=weight_dtype)
            generator = torch.Generator(device=device).manual_seed(42)
            with torch.no_grad():
                sample_images = sample_pipe(
                    test_prompts, num_inference_steps=30, guidance_scale=7.5,
                    generator=generator, output_type="pt",
                ).images  # returns tensor when output_type="pt"
            sample_history[global_step] = sample_images.cpu()
            unet.float()  # cast back to fp32 for training

            unet.train()
            print(f"         → Saved sample images at step {global_step}")

    if global_step >= num_steps:
        break

print("=" * 60)
print(f"Training complete! Final avg train loss: {np.mean(train_losses[-50:]):.4f}")

In [ ]:
# ============================================================
# Visualization: Training and Test Loss Curve + Learning Rate
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# --- Loss curve (train + test) ---
ax1.plot(range(1, len(train_losses)+1), train_losses, color='#457b9d', linewidth=1, alpha=0.3, label='Train (raw)')

if len(train_losses) > 20:
    window = min(30, len(train_losses) // 5)
    smoothed = np.convolve(train_losses, np.ones(window)/window, mode='valid')
    ax1.plot(range(window, len(train_losses)+1), smoothed, color='#457b9d', linewidth=2.5, label='Train (smoothed)')

# Overlay test loss
if test_losses:
    test_steps_plot = [x[0] for x in test_losses]
    test_vals_plot = [x[1] for x in test_losses]
    ax1.plot(test_steps_plot, test_vals_plot, color='#e63946', linewidth=2, marker='o',
             markersize=6, label='Test loss')

ax1.set_xlabel('Training Step')
ax1.set_ylabel('MSE Loss')
ax1.set_title('LoRA Fine-Tuning: Train and Test Loss', fontweight='bold', fontsize=14)
ax1.legend()

for ep in sample_history:
    ax1.axvline(x=ep, color='gray', alpha=0.3, linestyle='--')
    ax1.text(ep, ax1.get_ylim()[1]*0.95, f' step {ep}', fontsize=7, color='gray')

# --- Learning rate schedule ---
if lr_history:
    ax2.plot(range(1, len(lr_history)+1), lr_history, color='#e76f51', linewidth=2)
    ax2.axvline(x=warmup_steps, color='gray', alpha=0.5, linestyle='--', label=f'Warmup ends ({warmup_steps})')
    ax2.set_xlabel('Training Step')
    ax2.set_ylabel('Learning Rate')
    ax2.set_title('Learning Rate Schedule (Warmup + Cosine)', fontweight='bold', fontsize=14)
    ax2.legend()
    ax2.ticklabel_format(axis='y', style='scientific', scilimits=(-4,-4))

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Visualization: Generation Quality Over Training
# ============================================================

if sample_history:
    epochs_sorted = sorted(sample_history.keys())
    n_epochs_shown = len(epochs_sorted)
    n_prompts = len(test_prompts)

    fig, axes = plt.subplots(n_prompts, n_epochs_shown,
                             figsize=(4 * n_epochs_shown, 4.5 * n_prompts))
    if n_prompts == 1:
        axes = axes[np.newaxis, :]
    if n_epochs_shown == 1:
        axes = axes[:, np.newaxis]

    for col, step_num in enumerate(epochs_sorted):
        imgs = sample_history[step_num]
        for row in range(n_prompts):
            ax = axes[row, col]
            img = imgs[row].permute(1, 2, 0).numpy()
            ax.imshow(img.clip(0, 1))
            ax.axis('off')
            if row == 0:
                ax.set_title(f'Step {step_num}', fontweight='bold', fontsize=12)
            if col == 0:
                ax.set_ylabel(test_prompts[row][:30] + '...', fontsize=8, rotation=0,
                             labelpad=100, va='center')

    plt.suptitle('Generation Quality During LoRA Fine-Tuning\n(rows = prompts, columns = training steps)',
                fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No sample history available (training may not have run yet).")

## Step 6: Inference with the Fine-Tuned Model

Now let's use our LoRA-fine-tuned model to generate Pokemon images. We use the **`StableDiffusionPipeline`** for all inference — it handles the entire denoising loop, classifier-free guidance, and VAE decoding in one call.

We'll explore:
1. **Gallery generation** — various Pokemon prompts
2. **Before vs After** — comparing vanilla SD 1.5 with our fine-tuned model
3. **Guidance scale** — controlling how closely the model follows the prompt
4. **Denoising process** — watching a Pokemon emerge from noise step by step
5. **Seed variation** — same prompt, different random seeds

In [ ]:
# ============================================================
# Inference: Generate Pokemon Gallery
# ============================================================

unet.eval()

# Build inference pipeline with the fine-tuned model
inference_pipe = StableDiffusionPipeline(
    vae=vae, text_encoder=text_encoder, tokenizer=tokenizer,
    unet=unet, scheduler=noise_scheduler,
    safety_checker=None, feature_extractor=None,
    requires_safety_checker=False,
)

gallery_prompts = [
    "a drawing of a green pokemon with red eyes",
    "a red dragon breathing fire",
    "a blue water pokemon with fins and a tail",
    "a cute yellow electric mouse pokemon",
    "a purple ghost-like pokemon floating",
    "a white and blue ice creature",
    "a brown furry pokemon with big ears",
    "a pink small cute round creature",
]

print("Generating Pokemon images with fine-tuned LoRA model...")
generator = torch.Generator(device=device).manual_seed(42)
gallery_images = inference_pipe(
    gallery_prompts, num_inference_steps=30, guidance_scale=7.5,
    generator=generator,
).images

fig, axes = plt.subplots(2, 4, figsize=(20, 11))
for i, (img, prompt) in enumerate(zip(gallery_images, gallery_prompts)):
    ax = axes[i // 4, i % 4]
    ax.imshow(img)
    ax.set_title(prompt[:45], fontsize=9, wrap=True)
    ax.axis('off')

plt.suptitle('Generated Pokemon (LoRA Fine-Tuned SD 1.5, guidance_scale=7.5)',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Comparison: Before vs After LoRA Fine-Tuning
# ============================================================

compare_prompts = [
    "a drawing of a green pokemon with red eyes",
    "a blue water creature with fins",
    "a red fire-breathing dragon pokemon",
    "a cute yellow electric mouse pokemon",
]

print("Generating before/after comparison...")

# Generate with fine-tuned model
generator = torch.Generator(device=device).manual_seed(42)
finetuned_images = inference_pipe(
    compare_prompts, num_inference_steps=30, guidance_scale=7.5,
    generator=generator,
).images

# Disable LoRA to get baseline images (same weights, just LoRA turned off)
unet.disable_adapter_layers()
generator = torch.Generator(device=device).manual_seed(42)
baseline_compare = inference_pipe(
    compare_prompts, num_inference_steps=30, guidance_scale=7.5,
    generator=generator,
).images
unet.enable_adapter_layers()

# Display side by side
n = len(compare_prompts)
fig, axes = plt.subplots(2, n, figsize=(5 * n, 11))

for i in range(n):
    axes[0, i].imshow(baseline_compare[i])
    axes[0, i].set_title(compare_prompts[i][:40], fontsize=9, wrap=True)
    axes[0, i].axis('off')

    axes[1, i].imshow(finetuned_images[i])
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Before\n(Vanilla SD 1.5)', fontsize=12, rotation=0, labelpad=80, va='center')
axes[1, 0].set_ylabel('After\n(LoRA Fine-Tuned)', fontsize=12, rotation=0, labelpad=80, va='center')

plt.suptitle('Before vs After LoRA Fine-Tuning on Pokemon\n(same prompts, same seed)',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Experiment: Effect of Guidance Scale
# ============================================================

prompt = "a blue water pokemon with large eyes and fins"
guidance_scales = [1.0, 3.0, 5.0, 7.5, 12.0]

fig, axes = plt.subplots(1, len(guidance_scales), figsize=(5 * len(guidance_scales), 6))

print(f"Generating '{prompt}' at different guidance scales...")

for idx, gs in enumerate(guidance_scales):
    generator = torch.Generator(device=device).manual_seed(42)
    img = inference_pipe(
        prompt, num_inference_steps=30, guidance_scale=gs,
        generator=generator,
    ).images[0]
    axes[idx].imshow(img)
    axes[idx].set_title(f'scale = {gs}', fontweight='bold', fontsize=13)
    axes[idx].axis('off')

plt.suptitle(f'Classifier-Free Guidance Scale Comparison\nPrompt: "{prompt}"',
            fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("\nObservation:")
print("  scale=1:   Ignores the text (unconditional generation)")
print("  scale=3-5: Moderate text following")
print("  scale=7.5: Good balance of quality and prompt adherence")
print("  scale=12+: Very prompt-faithful but may over-saturate")

In [ ]:
# ============================================================
# Visualization: Step-by-Step Denoising Process
# ============================================================
# Watch as a Pokemon emerges from noise in latent space.
# We use the pipeline's callback_on_step_end to capture intermediate images.

from diffusers.image_processor import VaeImageProcessor

prompt_for_anim = "a red dragon pokemon with wings"
print(f"Generating denoising timeline for: '{prompt_for_anim}'")

denoising_snapshots = []
denoising_step_info = []

def capture_snapshot(pipe, step, timestep, callback_kwargs):
    """Callback to capture intermediate denoising images."""
    latents = callback_kwargs["latents"]
    with torch.no_grad():
        decoded = pipe.vae.decode(latents / pipe.vae.config.scaling_factor).sample
        img = (decoded[0].float() * 0.5 + 0.5).clamp(0, 1).cpu().permute(1, 2, 0).numpy()
    denoising_snapshots.append(img)
    denoising_step_info.append((step, timestep.item() if hasattr(timestep, 'item') else timestep))
    return callback_kwargs

generator = torch.Generator(device=device).manual_seed(123)
final_image = inference_pipe(
    prompt_for_anim,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator,
    callback_on_step_end=capture_snapshot,
).images[0]

# Show as filmstrip (select ~8 evenly spaced frames)
frames_to_show = list(range(0, len(denoising_snapshots), max(1, len(denoising_snapshots)//8)))
if (len(denoising_snapshots)-1) not in frames_to_show:
    frames_to_show.append(len(denoising_snapshots)-1)
n_frames = len(frames_to_show)

fig, axes = plt.subplots(1, n_frames, figsize=(3.5 * n_frames, 4.5))

for i, fidx in enumerate(frames_to_show):
    img = denoising_snapshots[fidx]
    step_num, t_val = denoising_step_info[fidx]
    axes[i].imshow(img)
    axes[i].set_title(f'Step {step_num+1}/{len(denoising_snapshots)}\nt={t_val:.0f}', fontsize=10)
    axes[i].axis('off')

plt.suptitle(f'Denoising Timeline: "{prompt_for_anim}"\nPure noise (left) → Generated Pokemon (right)',
            fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Experiment: Same Prompt, Different Random Seeds
# ============================================================

prompt_diversity = "a cute round blue pokemon"
n_samples = 6

fig, axes = plt.subplots(1, n_samples, figsize=(5 * n_samples, 6))

print(f"Generating {n_samples} variations of: '{prompt_diversity}'")

for i in range(n_samples):
    generator = torch.Generator(device=device).manual_seed(i * 42)
    img = inference_pipe(
        prompt_diversity, num_inference_steps=30, guidance_scale=7.5,
        generator=generator,
    ).images[0]
    axes[i].imshow(img)
    axes[i].set_title(f'seed = {i * 42}', fontsize=11)
    axes[i].axis('off')

plt.suptitle(f'Same Prompt, Different Seeds → Diverse Outputs\n"{prompt_diversity}"',
            fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("Each image starts from different random noise but follows the same text guidance.")

---

# Summary

## What This Notebook Covers

| Step | What We Did | Key Takeaway |
|------|------------|-------------|
| **Step 1** | Loaded the Pokemon BLIP Captions dataset | 833 images with simple text descriptions |
| **Step 2** | Loaded Stable Diffusion 1.5 via `StableDiffusionPipeline` | VAE + CLIP + U-Net + Scheduler in one call |
| **Step 3** | Injected LoRA adapters into the U-Net | Train <1% of parameters efficiently |
| **Step 4** | Understood Classifier-Free Guidance | CFG dropout during training, guidance scale at inference |
| **Step 5** | Trained the LoRA adapters | Latent-space diffusion loss with cosine LR |
| **Step 6** | Generated Pokemon with the fine-tuned model | Gallery, before/after, guidance scale, denoising timeline |

## Stable Diffusion Architecture

| Component | Detail |
|-----------|--------|
| **Base model** | Stable Diffusion 1.5 (`runwayml/stable-diffusion-v1-5`) |
| **Resolution** | 512×512 pixels (64×64×4 latent space) |
| **VAE** | Compresses images 8× spatially into latents |
| **Text encoder** | CLIP ViT-L/14 (frozen) |
| **U-Net** | ~860M params, fine-tuned with LoRA |
| **LoRA rank** | r=32 targeting attention + feed-forward layers |

## Key Simplifications in This Notebook

- **`StableDiffusionPipeline`** handles inference (tokenization, text encoding, denoising loop, CFG, VAE decoding) in one call
- **`callback_on_step_end`** captures intermediate denoising steps without writing manual sampling loops
- **`peft` library** injects LoRA with a few lines of code
- **`datasets` library** loads the Pokemon dataset with one function call